In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "global_settings.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
from src.global_settings import settings


In [ ]:
import glob
import os
import random

import torch
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from tqdm import tqdm

from src.activations import ActivationDataset

In [ ]:
BATCH_SIZE = 64
SEQ_LEN = 2048

TEXT_DATASET_PATH = str(settings.text_dataset_path)

df_text = pd.read_csv(TEXT_DATASET_PATH)

In [ ]:
N_BATCHES_TO_SAMPLE = 100

## Extracted Residual Streams:
- Data format: `dict` type in `.pt` files
- '/data/post_mlp_residual_stream/layer_1/batch_0.pt'
- <class 'dict'>
- dict_keys(['indices', 'activations', 'attention_mask'])

**indices**:
- torch.int64
- torch.Size([64])

**activations**:
- torch.float16
- torch.Size([64, 2048, 768])

**attention_mask**:
- torch.int64
- torch.Size([64, 2048])

In [ ]:
data_dir = "./data/post_mlp_residual_stream/layer_1"

file_paths = sorted(glob.glob(os.path.join(data_dir, "*.pt")))

file_path = file_paths[0]
filename = os.path.basename(file_path)
batch = torch.load(file_path, map_location="cpu")

batch_ids = batch["indices"]
batch_ts = batch["activations"]
batch_mask = batch["attention_mask"]

# Layer-1

In [ ]:
activation_dataset = ActivationDataset("./data/post_mlp_residual_stream/layer_1")

In [ ]:
loaded_batches = [activation_dataset[i] for i in tqdm(range(min(N_BATCHES_TO_SAMPLE, len(activation_dataset))))]

In [ ]:
full_activations = torch.cat(loaded_batches, dim=0)
num_total_activations = full_activations.shape[0]  # number of token activations

### cls token

In [ ]:
cls_vectors = []
for batch_tensor in tqdm(loaded_batches):
    reshaped_tensor = batch_tensor.view(BATCH_SIZE, SEQ_LEN, -1)
    cls_vector = reshaped_tensor[:, 0, :]

    cls_vectors.append(cls_vector)

all_cls_vectors = torch.cat(cls_vectors, dim=0)
all_cls_vectors = all_cls_vectors.numpy()

In [ ]:
num_samples = len(all_cls_vectors)
hue_labels = df_text.iloc[:num_samples]["label"]

pca = PCA(n_components=50, random_state=42)
cls_vectors_pca = pca.fit_transform(all_cls_vectors)

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
cls_vectors_2d = tsne.fit_transform(cls_vectors_pca)

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
sns.scatterplot(
    x=cls_vectors_2d[:, 0],
    y=cls_vectors_2d[:, 1],
    hue=hue_labels,
    palette={0: "royalblue", 1: "tomato"},
    s=15,
    alpha=0.7,
    legend="full"
)
plt.title(f"t-SNE of Residual Stream Vectors (CLS in Layer-1)")
plt.tight_layout()
plt.show()

## Layer-15

In [ ]:
activation_dataset = ActivationDataset("./data/post_mlp_residual_stream/layer_15")
loaded_batches = [activation_dataset[i] for i in tqdm(range(min(N_BATCHES_TO_SAMPLE, len(activation_dataset))))]

full_activations = torch.cat(loaded_batches, dim=0)
num_total_activations = full_activations.shape[0]

cls_vectors = []
for batch_tensor in tqdm(loaded_batches):
    reshaped_tensor = batch_tensor.view(BATCH_SIZE, SEQ_LEN, -1)
    cls_vector = reshaped_tensor[:, 0, :]

    cls_vectors.append(cls_vector)

all_cls_vectors = torch.cat(cls_vectors, dim=0)
all_cls_vectors = all_cls_vectors.numpy()
num_samples = len(all_cls_vectors)
hue_labels = df_text.iloc[:num_samples]["label"]

pca = PCA(n_components=50, random_state=42)
cls_vectors_pca = pca.fit_transform(all_cls_vectors)

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
cls_vectors_2d = tsne.fit_transform(cls_vectors_pca)

plt.style.use("seaborn-v0_8-whitegrid")
sns.scatterplot(
    x=cls_vectors_2d[:, 0],
    y=cls_vectors_2d[:, 1],
    hue=hue_labels,
    palette={0: "royalblue", 1: "tomato"},
    s=15,
    alpha=0.7,
    legend="full"
)
plt.title(f"t-SNE of Residual Stream Vectors (CLS in Layer-15)")
plt.tight_layout()
plt.show()

In [ ]:
N_CLUSTERS = 5

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init="auto")
cluster_labels = kmeans.fit_predict(cls_vectors_2d)

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
sns.scatterplot(
    x=cls_vectors_2d[:, 0],
    y=cls_vectors_2d[:, 1],
    hue=cluster_labels,
    palette=sns.color_palette("turbo", N_CLUSTERS),
    s=15,
    alpha=0.7,
    legend="full"
)
plt.title(f"Clustered t-SNE of Residual Stream Vectors (k={N_CLUSTERS}, Layer-15)")
plt.tight_layout()
plt.show()

In [ ]:
for i in range(N_CLUSTERS):
    print(f"\n\n{'='*25} Cluster {i} {'='*25}\n")
    ids_in_cluster = np.where(cluster_labels == i)[0]
    sample_indices = np.random.choice(ids_in_cluster, size=min(5, len(ids_in_cluster)), replace=False)
    for idx in sample_indices:
        text = df_text.iloc[idx]["text"]
        print(f"--- Sample: {idx} ---")
        print(f"{text[:500]}...\n")